# Notebook 2: Data Structures, Files, NumPy and pandas

**Audience:** students on MSc Advanced Machine Learning, MSc Artificial Intelligence, and MSc Data Science who have completed Notebook 1  
**Suggested time:** 90–120 minutes

## Learning outcomes
By the end of this notebook you should be able to:

- import and use Python modules;
- work with paths and text/CSV files;
- read and write CSV data with Python's standard library;
- create and manipulate NumPy arrays;
- create and inspect pandas `Series` and `DataFrame` objects;
- filter, sort, summarise and group tabular data;
- handle simple missing values;
- save processed data back to CSV.

This notebook creates all of its own example files, so it does not need an internet connection or separate dataset download.

## 1. Modules and imports

Python's core language is deliberately small. **Modules** contain reusable functions, classes and constants.

Some modules are part of the Python standard library. Others, such as NumPy and pandas, are third-party packages installed separately.

In [2]:
import math

print(math.sqrt(81))
print(math.pi)

9.0
3.141592653589793


You can import individual names, or use an alias.

In [1]:
from statistics import mean
from pathlib import Path

print(mean([10, 20, 30]))
print(Path.cwd())

20
h:\msc-introductory-python-main


The conventional aliases for NumPy and pandas are `np` and `pd`.

In [3]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

NumPy version: 2.3.5
pandas version: 2.3.3


You can inspect a module with `dir(...)`, or ask Jupyter for help by writing a `?` after a function name, for example `pd.read_csv?`. We will avoid opening the interactive help panel during **Run All**, but try it yourself.

### Exercise 1: Use a module
Import the standard-library `random` module and use `random.seed(42)` followed by `random.randint(1, 10)` to generate a reproducible random integer.

In [7]:
# TODO: import random, set the seed, and generate an integer.
import random 
random.seed(42) 
print( random.randint(1,10) ) 

2


## 2. Paths and files

We will keep generated files in a folder called `data_structures_files`.

In [8]:
from pathlib import Path

data_dir = Path("data_structures_files")
data_dir.mkdir(exist_ok=True)

print("Working folder:", data_dir.resolve())

Working folder: \\myfiles.campus.bath.ac.uk\tl958\msc-introductory-python-main\data_structures_files


Create and read a plain-text file.

In [9]:
message_path = data_dir / "message.txt"

message_path.write_text(
    "row_count=6\n"
    "source=synthetic classroom data\n"
    "status=ready\n",
    encoding="utf-8",
)

print(message_path.read_text(encoding="utf-8"))

row_count=6
source=synthetic classroom data
status=ready



### Exercise 2: Parse a simple text file
Read `message.txt` line by line. Split each line at `=` and build a dictionary.

In [19]:
metadata = {}

# TODO: read message_path and populate metadata.
ori = message_path.read_text()
key = ""
value = ""
isValue = False  
for i in ori : 
    # print( i )
    if( i == '\n' ) :
        # print( key ) 
        metadata[key] = value 
        key = ""
        value = ""
        isValue = False
    elif( i == '='):
        isValue = True
    elif( isValue ) :
        value += i 
    else : 
        key += i 

        
print(metadata)

{'row_count': '6', 'source': 'synthetic classroom data', 'status': 'ready'}


## 3. CSV files with the standard library

CSV means **comma-separated values**. It is a common plain-text format for tabular data.

Python's built-in `csv` module is useful when you want direct control over CSV reading or writing.

In [20]:
import csv

csv_path = data_dir / "students.csv"

rows = [
    ["student_id", "programme", "hours_studied", "quiz_mark"],
    ["s001", "Data Science", 4.5, 72],
    ["s002", "AI", 3.0, 65],
    ["s003", "Data Science", 6.0, 81],
    ["s004", "AI", 2.5, 58],
    ["s005", "Data Science", 5.0, 77],
    ["s006", "AI", 4.0, 69],
]

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("Wrote:", csv_path)

Wrote: data_structures_files\students.csv


In [21]:
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

['student_id', 'programme', 'hours_studied', 'quiz_mark']
['s001', 'Data Science', '4.5', '72']
['s002', 'AI', '3.0', '65']
['s003', 'Data Science', '6.0', '81']
['s004', 'AI', '2.5', '58']
['s005', 'Data Science', '5.0', '77']
['s006', 'AI', '4.0', '69']


`csv.DictReader` uses the header row as dictionary keys.

In [22]:
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    standard_library_records = list(reader)

print(standard_library_records[0])
print("Number of records:", len(standard_library_records))

{'student_id': 's001', 'programme': 'Data Science', 'hours_studied': '4.5', 'quiz_mark': '72'}
Number of records: 6


Notice that CSV values read by the standard library are strings unless you explicitly convert them.

In [23]:
first_record = standard_library_records[0]
print(first_record["quiz_mark"], type(first_record["quiz_mark"]))

quiz_mark_as_int = int(first_record["quiz_mark"])
print(quiz_mark_as_int, type(quiz_mark_as_int))

72 <class 'str'>
72 <class 'int'>


### Exercise 3: CSV calculation
Using `standard_library_records`, calculate the average `quiz_mark`. Remember to convert each mark to an integer.

In [24]:
# TODO: calculate the average quiz mark from standard_library_records.
average_quiz_mark = None
totalmark = 0 
for i in standard_library_records : 
    totalmark += int(first_record["quiz_mark"])

average_quiz_mark = totalmark / len(standard_library_records)
print("Average:", average_quiz_mark)

Average: 72.0


## 4. NumPy arrays

NumPy provides efficient **n-dimensional arrays** and numerical operations.

A NumPy array usually contains values of one data type and supports **vectorised** operations: operations applied to many values without writing an explicit Python loop.

In [25]:
numbers = np.array([10, 20, 30, 40, 50])

print(numbers)
print(type(numbers))
print("shape:", numbers.shape)
print("dtype:", numbers.dtype)
print("dimensions:", numbers.ndim)

[10 20 30 40 50]
<class 'numpy.ndarray'>
shape: (5,)
dtype: int64
dimensions: 1


Indexing and slicing look similar to Python lists.

In [26]:
print(numbers[0])
print(numbers[-1])
print(numbers[1:4])

10
50
[20 30 40]


Vectorised arithmetic applies element by element.

In [27]:
print(numbers + 5)
print(numbers * 2)
print(numbers / 10)

[15 25 35 45 55]
[ 20  40  60  80 100]
[1. 2. 3. 4. 5.]


NumPy includes many aggregation functions.

In [28]:
print("sum:", np.sum(numbers))
print("mean:", np.mean(numbers))
print("median:", np.median(numbers))
print("min:", np.min(numbers))
print("max:", np.max(numbers))
print("standard deviation:", np.std(numbers))

sum: 150
mean: 30.0
median: 30.0
min: 10
max: 50
standard deviation: 14.142135623730951


Two-dimensional arrays can represent tables or matrices.

In [29]:
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

print(matrix)
print("shape:", matrix.shape)
print("item at row 0, column 1:", matrix[0, 1])
print("second column:", matrix[:, 1])

[[1 2 3]
 [4 5 6]
 [7 8 9]]
shape: (3, 3)
item at row 0, column 1: 2
second column: [2 5 8]


Boolean masks select values that satisfy a condition.

In [30]:
scores = np.array([45, 61, 72, 38, 55, 89])

passing_mask = scores >= 50
print(passing_mask)
print(scores[passing_mask])

[False  True  True False  True  True]
[61 72 55 89]


### Exercise 4: NumPy practice
For the array below:
- calculate its mean;
- select values greater than the mean;
- subtract the mean from every value;
- reshape it into 2 rows and 3 columns.

In [44]:
values = np.array([2, 4, 6, 8, 10, 12])

# TODO: perform the four operations.
print( np.mean( values ))
for i in values : 
    if( i > np.mean( values ) ): 
        print( i )

print( values - np.mean( values ) )
matrix= [] 
matrix = values[0:3]
matrix = np.vstack( [matrix , values[3:6]])

print( matrix)
print(values)

7.0
8
10
12
[-5. -3. -1.  1.  3.  5.]
[[ 2  4  6]
 [ 8 10 12]]
[ 2  4  6  8 10 12]


## 5. pandas Series and DataFrames

pandas is designed for labelled, tabular data.

A `Series` is one-dimensional. A `DataFrame` is a two-dimensional table with labelled columns and an index.

In [45]:
temperatures = pd.Series([12.5, 14.2, 13.8], name="temperature_c")
print(temperatures)

0    12.5
1    14.2
2    13.8
Name: temperature_c, dtype: float64


In [46]:
example_df = pd.DataFrame({
    "city": ["Bath", "Bristol", "London"],
    "temperature_c": [12.5, 14.2, 15.1],
    "raining": [True, False, False],
})

example_df

,city,temperature_c,raining
0,Bath,12.5,True
1,Bristol,14.2,False
2,London,15.1,False


## 6. Reading CSV with pandas

`pd.read_csv(...)` reads a CSV file directly into a DataFrame. pandas also attempts to infer useful data types.

In [47]:
df = pd.read_csv(csv_path)
df

,student_id,programme,hours_studied,quiz_mark
0,s001,Data Science,4.5,72
1,s002,AI,3.0,65
2,s003,Data Science,6.0,81
3,s004,AI,2.5,58
4,s005,Data Science,5.0,77
5,s006,AI,4.0,69


In [48]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(6, 4)
['student_id', 'programme', 'hours_studied', 'quiz_mark']
student_id        object
programme         object
hours_studied    float64
quiz_mark          int64
dtype: object


Useful first inspection methods include `head()`, `tail()`, `info()` and `describe()`.

In [49]:
display(df.head())
display(df.select_dtypes(include="number").describe())

,student_id,programme,hours_studied,quiz_mark
0,s001,Data Science,4.5,72
1,s002,AI,3.0,65
2,s003,Data Science,6.0,81
3,s004,AI,2.5,58
4,s005,Data Science,5.0,77


,hours_studied,quiz_mark
count,6.000000,6.000000
mean,4.166667,70.333333
std,1.290994,8.286535
min,2.500000,58.000000
25%,3.250000,66.000000
50%,4.250000,70.500000
75%,4.875000,75.750000
max,6.000000,81.000000


Select a single column with brackets. Selecting multiple columns requires a list of column names.

In [61]:
print(df["quiz_mark"])

display(df[["student_id", "quiz_mark"]])

0    72
1    65
2    81
3    58
4    77
5    69
Name: quiz_mark, dtype: int64


,student_id,quiz_mark
0,s001,72
1,s002,65
2,s003,81
3,s004,58
4,s005,77
5,s006,69


`.loc` selects by labels and conditions. `.iloc` selects by integer position.

In [51]:
display(df.loc[df["quiz_mark"] >= 70, ["student_id", "quiz_mark"]])
display(df.iloc[0:3, 0:2])

,student_id,quiz_mark
0,s001,72
2,s003,81
4,s005,77


,student_id,programme
0,s001,Data Science
1,s002,AI
2,s003,Data Science


### Exercise 5: Inspect and filter
Using `df`:
- show the final three rows;
- select only `programme` and `hours_studied`;
- filter to students who studied at least 4 hours;
- filter to students whose quiz mark is at least 70.

In [66]:
# TODO: add your pandas expressions.
display( df.tail(3))
display(df[["programme", "hours_studied"]])

display(df.loc[df["hours_studied"] >= 4])
display(df.loc[df["quiz_mark"] >= 70])

display(df.head(2))

,student_id,programme,hours_studied,quiz_mark
3,s004,AI,2.5,58
4,s005,Data Science,5.0,77
5,s006,AI,4.0,69


,programme,hours_studied
0,Data Science,4.5
1,AI,3.0
2,Data Science,6.0
3,AI,2.5
4,Data Science,5.0
5,AI,4.0


,student_id,programme,hours_studied,quiz_mark
0,s001,Data Science,4.5,72
2,s003,Data Science,6.0,81
4,s005,Data Science,5.0,77
5,s006,AI,4.0,69


,student_id,programme,hours_studied,quiz_mark
0,s001,Data Science,4.5,72
2,s003,Data Science,6.0,81
4,s005,Data Science,5.0,77


,student_id,programme,hours_studied,quiz_mark
0,s001,Data Science,4.5,72
1,s002,AI,3.0,65


## 7. Creating and transforming columns

New columns can be created from existing columns using vectorised expressions.

In [67]:
df["hours_per_mark"] = df["hours_studied"] / df["quiz_mark"]
df["passed_quiz"] = df["quiz_mark"] >= 50

df

,student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz
0,s001,Data Science,4.5,72,0.062500,True
1,s002,AI,3.0,65,0.046154,True
2,s003,Data Science,6.0,81,0.074074,True
3,s004,AI,2.5,58,0.043103,True
4,s005,Data Science,5.0,77,0.064935,True
5,s006,AI,4.0,69,0.057971,True


Use `sort_values()` to order rows.

In [68]:
display(df.sort_values("quiz_mark", ascending=False))

,student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz
2,s003,Data Science,6.0,81,0.074074,True
4,s005,Data Science,5.0,77,0.064935,True
0,s001,Data Science,4.5,72,0.062500,True
5,s006,AI,4.0,69,0.057971,True
1,s002,AI,3.0,65,0.046154,True
3,s004,AI,2.5,58,0.043103,True


### Exercise 6: Add a column
Create a column called `mark_above_60` containing the number of marks above (or below) 60 for each student.

Then sort the DataFrame from the fewest hours studied to the most.

In [73]:
# TODO: create mark_above_60 and sort the data.
df["mark_above_60"] = df["quiz_mark"] >= 60
display(df.sort_values("hours_studied", ascending=True))

display(df.head())

,student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz,mark_above_60
3,s004,AI,2.5,58,0.043103,True,False
1,s002,AI,3.0,65,0.046154,True,True
5,s006,AI,4.0,69,0.057971,True,True
0,s001,Data Science,4.5,72,0.062500,True,True
4,s005,Data Science,5.0,77,0.064935,True,True
2,s003,Data Science,6.0,81,0.074074,True,True


,student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz,mark_above_60
0,s001,Data Science,4.5,72,0.062500,True,True
1,s002,AI,3.0,65,0.046154,True,True
2,s003,Data Science,6.0,81,0.074074,True,True
3,s004,AI,2.5,58,0.043103,True,False
4,s005,Data Science,5.0,77,0.064935,True,True


## 8. Grouping and aggregation

`groupby()` is one of pandas' most important tools. It follows a **split → apply → combine** pattern.

In [76]:
programme_summary = (
    df.groupby("programme")
      .agg(
          students=("student_id", "count"),
          mean_hours=("hours_studied", "mean"),
          mean_mark=("quiz_mark", "mean"),
          highest_mark=("quiz_mark", "max"),
      )
)

programme_summary

,students,mean_hours,mean_mark,highest_mark
programme,,,,
AI,3,3.166667,64.000000,69
Data Science,3,5.166667,76.666667,81


### Exercise 7: Grouped summary
Group by `programme` and calculate:
- minimum quiz mark;
- median quiz mark;
- maximum hours studied.

In [78]:
# TODO: create a grouped summary.

exercise_summary = pd.DataFrame()
exercise_summary = (
    df.groupby("programme")
      .agg(
          mean_hours=("quiz_mark", "median"),
          mean_mark=("quiz_mark", "min"),
          highest_mark=("hours_studied", "max"),
      )
)

exercise_summary

,mean_hours,mean_mark,highest_mark
programme,,,
AI,65.0,58,4.0
Data Science,77.0,72,6.0


## 9. Missing values

Real data is often incomplete. pandas represents many missing values as `NaN`.

The next cell creates a small example dataset with missing values.

In [79]:
messy_df = pd.DataFrame({
    "student_id": ["s101", "s102", "s103", "s104"],
    "hours_studied": [4.0, np.nan, 5.5, 3.0],
    "quiz_mark": [70, 62, np.nan, 55],
})

messy_df

,student_id,hours_studied,quiz_mark
0,s101,4.0,70.0
1,s102,NaN,62.0
2,s103,5.5,NaN
3,s104,3.0,55.0


In [82]:
print(messy_df.isna())
print("\nMissing values per column:")
print(messy_df.isna().sum())

   student_id  hours_studied  quiz_mark
0       False          False      False
1       False           True      False
2       False          False       True
3       False          False      False

Missing values per column:
student_id       0
hours_studied    1
quiz_mark        1
dtype: int64


There is no single correct way to handle missing data. The right choice depends on why values are missing and how the data will be used.

For a simple demonstration, we can fill a missing numeric value with the column median.

In [83]:
filled_df = messy_df.copy()

filled_df["hours_studied"] = filled_df["hours_studied"].fillna(
    filled_df["hours_studied"].median()
)

filled_df

,student_id,hours_studied,quiz_mark
0,s101,4.0,70.0
1,s102,4.0,62.0
2,s103,5.5,NaN
3,s104,3.0,55.0


Rows can also be removed with `dropna()`, but dropping data blindly can introduce problems. Always understand what is missing before choosing a strategy.

In [84]:
complete_rows = messy_df.dropna()
complete_rows

,student_id,hours_studied,quiz_mark
0,s101,4.0,70.0
3,s104,3.0,55.0


### Exercise 8: Missing values
Create a copy of `messy_df` called `cleaned_df`. Fill the missing `quiz_mark` using the median quiz mark, and print the number of missing values remaining in each column.

In [95]:
cleaned_df = messy_df.copy()
display(cleaned_df)
# TODO: fill the missing quiz_mark.
cleaned_df["quiz_mark"] = filled_df["quiz_mark"].fillna(
    filled_df["quiz_mark"].median()
)

display(cleaned_df)
# print(cleaned_df.isna().sum())

,student_id,hours_studied,quiz_mark
0,s101,4.0,70.0
1,s102,NaN,62.0
2,s103,5.5,NaN
3,s104,3.0,55.0


,student_id,hours_studied,quiz_mark
0,s101,4.0,70.0
1,s102,NaN,62.0
2,s103,5.5,62.0
3,s104,3.0,55.0


## 10. Writing processed data

Use `DataFrame.to_csv(...)` to save a DataFrame. `index=False` avoids writing the DataFrame's row index as an extra CSV column.

In [96]:
processed_path = data_dir / "students_processed.csv"
df.to_csv(processed_path, index=False)

print("Saved:", processed_path)
print(processed_path.read_text(encoding="utf-8").splitlines()[:3])

Saved: data_structures_files\students_processed.csv
['student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz,mark_above_60', 's001,Data Science,4.5,72,0.0625,True,True', 's002,AI,3.0,65,0.046153846153846156,True,True']


Read the file back to verify it.

In [97]:
round_trip_df = pd.read_csv(processed_path)
round_trip_df.head()

,student_id,programme,hours_studied,quiz_mark,hours_per_mark,passed_quiz,mark_above_60
0,s001,Data Science,4.5,72,0.062500,True,True
1,s002,AI,3.0,65,0.046154,True,True
2,s003,Data Science,6.0,81,0.074074,True,True
3,s004,AI,2.5,58,0.043103,True,False
4,s005,Data Science,5.0,77,0.064935,True,True


## 11. A tiny custom module

A Python file ending in `.py` can itself be a module.

The next cell writes a very small module to disk. In a normal project you would usually create the file in your editor rather than from inside a notebook.

In [99]:
module_path = data_dir / "data_helpers.py"

module_path.write_text(
    'def percentage(part, whole):\n'
    '    return 100 * part / whole\n'
    '\n'
    'def centre(value, mean_value):\n'
    '    return value - mean_value\n',
    encoding="utf-8",
)

print(module_path.read_text(encoding="utf-8"))

def percentage(part, whole):
    return 100 * part / whole

def centre(value, mean_value):
    return value - mean_value



To import a module from a custom path in a robust notebook example, we can use `importlib`.

In [100]:
import importlib.util

spec = importlib.util.spec_from_file_location("data_helpers", module_path)
data_helpers = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_helpers)

print(data_helpers.percentage(18, 24))
print(data_helpers.centre(72, 65))

75.0
7


# Final mini-challenge: Analyse a CSV

Starting from `df`, create a small analysis that:

1. keeps only students who studied at least 4 hours;
2. creates a `mark_per_hour` column;
3. sorts by `quiz_mark` from highest to lowest;
4. calculates the mean mark for each programme;
5. saves the filtered table to `data_structures_files/focused_students.csv`.

Use a `.copy()` when creating the filtered DataFrame so you can modify it safely.

In [ ]:
# TODO: implement the mini-challenge.
focused = df.copy()
display(focused.head())

# Recap

You have moved from core Python structures to tools used constantly in data science:

- **modules** organise reusable code;
- **CSV** stores tabular data in plain text;
- **NumPy arrays** support efficient numerical computation;
- **pandas DataFrames** provide labelled tables, filtering, grouping and file I/O.

The next notebook uses pandas and NumPy ideas inside a scikit-learn machine-learning workflow.

# Solutions

Solutions to the exercises are collected here so you can attempt each task before checking your answer. These are example solutions: there is often more than one correct way to solve a problem.


## Exercise 1: Use a module


In [ ]:
import random

random.seed(42)
random_number = random.randint(1, 10)
print(random_number)

## Exercise 2: Parse a simple text file


In [ ]:
metadata = {}

with open(message_path, "r", encoding="utf-8") as file:
    for line in file:
        key, value = line.strip().split("=", maxsplit=1)
        metadata[key] = value

print(metadata)

## Exercise 3: CSV calculation


In [ ]:
marks = [int(record["quiz_mark"]) for record in standard_library_records]
average_quiz_mark = sum(marks) / len(marks)
print("Average:", average_quiz_mark)

## Exercise 4: NumPy practice


In [ ]:
values = np.array([2, 4, 6, 8, 10, 12])

values_mean = values.mean()
print("mean:", values_mean)
print("above mean:", values[values > values_mean])
print("centred:", values - values_mean)
print("reshaped:\n", values.reshape(2, 3))

## Exercise 5: Inspect and filter


In [ ]:
display(df.tail(3))
display(df[["programme", "hours_studied"]])
display(df.loc[df["hours_studied"] >= 4])
display(df.loc[df["quiz_mark"] >= 70])

## Exercise 6: Add a column


In [ ]:
df["mark_above_60"] = df["quiz_mark"] - 60
display(df.sort_values("hours_studied"))

## Exercise 7: Grouped summary


In [ ]:
exercise_summary = (
    df.groupby("programme")
      .agg(
          minimum_mark=("quiz_mark", "min"),
          median_mark=("quiz_mark", "median"),
          maximum_hours=("hours_studied", "max"),
      )
)

exercise_summary

## Exercise 8: Missing values


In [ ]:
cleaned_df = messy_df.copy()
cleaned_df["quiz_mark"] = cleaned_df["quiz_mark"].fillna(
    cleaned_df["quiz_mark"].median()
)

print(cleaned_df.isna().sum())

## Final mini-challenge: Analyse a CSV


In [ ]:
focused = df.loc[df["hours_studied"] >= 4].copy()
focused["mark_per_hour"] = focused["quiz_mark"] / focused["hours_studied"]
focused = focused.sort_values("quiz_mark", ascending=False)

mean_by_programme = focused.groupby("programme")["quiz_mark"].mean()

focused_path = data_dir / "focused_students.csv"
focused.to_csv(focused_path, index=False)

display(focused)
print("\nMean mark by programme:")
print(mean_by_programme)
print("\nSaved:", focused_path)